In [1]:
import sys
sys.path.insert(0, "src")  # /workspace/src

from patent_train import TrainingRunner, TrainConfig

In [2]:
SEARCH = False   # True=fast-fail 짧은 런(2 epoch로 유도) / False=풀런(아래 epochs 사용)

cfg = TrainConfig(
    backbone="axenc_tapt",   # TAPT 위해 백본 변경
    loss="focal",
    loss_params={"alpha": 0.25, "gamma": 2},
    max_len=512,
    eff_batch=128,          # 배치 재현 파라미터
    micro_batch=128,        
    eval_micro_batch=512,
    learning_rate=4.8e-4,   # 확정 레시피 lr
    weight_decay=0.01,
    warmup_ratio=0.1,
    epochs=12,
    early_stop_epochs=2,    # 개선 없이 견디는 에폭 수(eval 횟수 환산은 runner가 처리)
    notebook_name="13_02_TAPT_Train.ipynb",   # wandb code saving
    tag="modernbert-patent-len512-tapt",
    run_name="axenc_len512_tapt_train",
    repo_final="ingyoun/A.X-patent-len512-tapt",
    out_path="/workspace/output/modernbert-len512-tapt",
    search=SEARCH,
)

print("run_name:", cfg.run_name, "| epochs:", cfg.epochs, "| grad_accum:", cfg.grad_accum)

run_name: axenc_len512_tapt_train | epochs: 12 | grad_accum: 1


## 구성 — 데이터·모델

토크나이저+원본 로드 → `max_len` 절단(캐시) → 분류기 구성. 단계를 나눠 중간 점검·부분 재실행이 가능하다.

In [3]:
runner = TrainingRunner(cfg)

In [4]:
runner.load_data()        # 토크나이저 + 원본 데이터셋(prep 캐시 있으면 원본 생략)
runner.data.raw

config.json:   0%|          | 0.00/1.94k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201616
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11244
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11132
    })
})

In [5]:
runner.prepare_data()     # max_len 절단 → prep 캐시
runner.data.dataset

Map:   0%|          | 0/201616 [00:00<?, ? examples/s]

Map:   0%|          | 0/11244 [00:00<?, ? examples/s]

Map:   0%|          | 0/11132 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/201616 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11244 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11132 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 201616
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11244
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11132
    })
})

In [6]:
runner.load_model()

model.safetensors:   0%|          | 0.00/598M [00:00<?, ?B/s]

[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertForSequenceClassification is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: ingyoun/A.X-patent-tapt-mlm
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 훈련

In [7]:
runner.build_trainer()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


[schedule] 1576 step/epoch | eval·save 788 step마다(2회/epoch) | early stop 2 epoch(patience=4 eval)


In [8]:
runner.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
788,0.001392,0.000902,0.638523,0.599728,0.565103,0.289436,0.659699
1576,0.000919,0.000685,0.726397,0.697534,0.694833,0.150647,0.715187
2364,0.000640,0.000537,0.770541,0.752976,0.747611,0.115972,0.758450
3152,0.000557,0.000522,0.777598,0.762356,0.757946,0.106630,0.764036
3940,0.000506,0.000486,0.792618,0.781665,0.781541,0.086148,0.773863
4728,0.000508,0.000491,0.795677,0.782424,0.792230,0.066565,0.780012
5516,0.000456,0.000474,0.807199,0.797334,0.802959,0.066206,0.782789
6304,0.000464,0.000448,0.810654,0.802752,0.802840,0.071685,0.785376
7092,0.000395,0.000455,0.820059,0.812578,0.824710,0.041412,0.791539
7880,0.000392,0.000452,0.814769,0.807093,0.820797,0.044107,0.788298


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 평가 · 메트릭 저장 · push

모델 가중치는 로컬에 두지 않고 Hub로만 올린다(팟을 지우면 로컬 사본은 사라진다). 로컬 사본이 필요하면 `runner.save_model()`.

In [9]:
test_metrics = runner.evaluate("test")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

[transformers] early stopping required metric_for_best_model, but did not find eval_micro_f1 so early stopping is disabled


Training Loss,Validation Loss,Step,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
0.000021,0.000889,18912,0.857248,0.854534,0.872390,0.011917,0.819835


test_loss: 0.0008891097968444228
test_micro_f1: 0.8572477064220183
test_macro_f1: 0.8545344498455583
test_sample_f1: 0.8723901908400309
test_empty_rate: 0.01191746709356101
test_anchor_weighted_f1: 0.8198348406285512


In [10]:
runner.save_metrics()     # runner.metrics(split 전체) → {tag}_metrics.json

[save] /workspace/output/modernbert-len512-tapt/modernbert-patent-len512-tapt_metrics.json  splits=['test']


In [11]:
runner.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

[push] ingyoun/A.X-patent-len512-tapt


## val·test 로짓 덤프

`logits_{tag}_{split}.npy`를 `out_path` 상위(`/workspace/output/`)에 저장

In [12]:
runner.predict_logits("val")
runner.predict_logits("test")

[dump] /workspace/output/logits_modernbert-patent-len512-tapt_val.npy  shape=(11132, 188)


[dump] /workspace/output/logits_modernbert-patent-len512-tapt_test.npy  shape=(11244, 188)


array([[ -6.6875  ,  -1.484375,  -6.46875 , ...,  -8.0625  ,  -5.84375 ,
         -7.375   ],
       [ -9.5     ,  -9.875   , -10.75    , ...,  -9.5625  ,  -5.5     ,
         -7.4375  ],
       [ -8.9375  ,  -8.5     ,  -5.8125  , ...,  -6.28125 ,  -7.46875 ,
         -7.9375  ],
       ...,
       [ -6.75    ,  -7.96875 ,  -7.96875 , ...,  -7.4375  ,  -7.125   ,
         -7.8125  ],
       [ -8.75    ,  -7.90625 , -10.9375  , ...,  -9.5     ,  -8.125   ,
         -9.9375  ],
       [ -7.15625 ,  -7.25    ,  -6.5625  , ...,  -6.03125 ,  -5.34375 ,
         -6.03125 ]], shape=(11244, 188), dtype=float32)